In [ ]:
%%sql -r dataframe_5
SELECT CURRENT_ROLE(), CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE();

In [ ]:
%%sql -r dataframe_8
-- Verify Snowflake sees the file
LIST @public.mmg.s3_stage;          -- tests stage + integration + schema access

In [ ]:
%%sql -r dataframe_2
-- create table for each csv file in S3
CREATE OR REPLACE TABLE public.bronze.zip_code
  USING TEMPLATE (
    SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
    FROM TABLE(
      INFER_SCHEMA(
        LOCATION => '@public.bronze.s3_stage/zip-code/zip-code-2020-2023.csv',
        FILE_FORMAT => 'public.bronze.csv_ff'
      )
    )
  );

In [ ]:
%%sql -r dataframe_13
CREATE OR REPLACE PIPE public.bronze.load_zip_code_pipe
  AUTO_INGEST = TRUE
AS
  COPY INTO public.bronze.zip_code
    FROM @public.bronze.s3_stage/zip-code/
    FILE_FORMAT = public.bronze.csv_ff
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE

In [ ]:
%%sql -r dataframe_14
show pipes

In [ ]:
-- SELECT SYSTEM$PIPE_STATUS('public.bronze.load_census_tract_pipe');
-- SELECT SYSTEM$PIPE_STATUS('public.bronze.load_county_pipe');
SELECT SYSTEM$PIPE_STATUS('public.bronze.load_zip_code_pipe');

In [ ]:
ALTER PIPE public.bronze.load_zip_code_pipe REFRESH;

In [ ]:
%%sql -r dataframe_3
-- TRUNCATE TABLE public.mmg.census_tract;
-- TRUNCATE TABLE public.mmg.county;

In [ ]:
%%sql -r dataframe_11
SELECT * FROM public.bronze.census_tract LIMIT 5
-- SELECT * FROM public.mmg.county LIMIT 5
-- SELECT * FROM public.mmg.zip_code LIMIT 5